# Mask Quality Analysis — YOLOv8 vs YOLOv8+SAM2

Two analyses:
1. **Qualitative** — regional overview and individual boulder comparisons (YOLO mask vs SAM2 mask on same image crop)
2. **Mask regularity metrics** — compactness and convexity distributions

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")
if "/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src" in sys.path:
    sys.path.remove("/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from pathlib import Path
from tqdm import tqdm
from shapely.geometry import box

from rastertools_BOULDERING import metadata as raster_metadata

work_dir      = Path("/scratch/users/cayleigh/tmp/YOLOv8BeyondEarth")
in_raster     = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")
yolo_output_dir = work_dir / "exp_yolo_256"
sam2_output_dir = work_dir / "exp_sam2_256"

In [ ]:
# Load shapefiles and apply area filter
def load_shapefile(output_dir, glob="*-downscaled-mask-nms.shp"):
    shps = sorted(output_dir.glob(glob))
    assert shps, f"No shapefiles found in {output_dir}"
    gdfs = [gpd.read_file(p) for p in shps]
    gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf["poly_area"] = gdf.geometry.area
    return gdf

gdf_yolo = load_shapefile(yolo_output_dir)
gdf_sam2 = load_shapefile(sam2_output_dir)

res = raster_metadata.get_resolution(in_raster)[0]
areal_threshold = (res ** 2) * (4.74 ** 2)

gdf_yolo = gdf_yolo[gdf_yolo["poly_area"] >= areal_threshold].reset_index(drop=True)
gdf_sam2 = gdf_sam2[gdf_sam2["poly_area"] >= areal_threshold].reset_index(drop=True)

print(f"YOLO: {len(gdf_yolo)} detections")
print(f"SAM2: {len(gdf_sam2)} detections")

## Part 1: Qualitative Comparison
### 1a. Regional overview — same 150m × 150m patch with both pipelines overlaid

In [ ]:
# Pick a central patch of the raster
with rasterio.open(in_raster) as src:
    b = src.bounds
    cx = (b.left + b.right) / 2
    cy = (b.bottom + b.top) / 2

region_size = 150  # metres
region_bounds = (cx - region_size/2, cy - region_size/2,
                 cx + region_size/2, cy + region_size/2)
region_box = box(*region_bounds)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

with rasterio.open(in_raster) as src:
    window = from_bounds(*region_bounds, src.transform)
    img = src.read(1, window=window).astype(float)
    p2, p98 = np.percentile(img, [2, 98])
    img_norm = np.clip((img - p2) / (p98 - p2 + 1e-6), 0, 1)
    win_transform = src.window_transform(window)

    for ax, (gdf, label, color) in zip(axes, [
        (gdf_yolo, "YOLOv8",       "steelblue"),
        (gdf_sam2, "YOLOv8+SAM2",  "tomato"),
    ]):
        ax.imshow(img_norm, cmap="gray", origin="upper")
        local = gdf[gdf.geometry.intersects(region_box)]
        for geom in local.geometry:
            coords = np.array(geom.exterior.coords)
            px = np.array([~win_transform * (x, y) for x, y in coords])
            ax.plot(px[:, 0], px[:, 1], color=color, linewidth=0.9, alpha=0.85)
        ax.set_title(f"{label}\n{len(local)} detections in region", fontsize=11)
        ax.axis("off")

plt.suptitle("Regional comparison — 150 m × 150 m patch", fontsize=13)
plt.tight_layout()
plt.savefig("qual_regional_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 1b. Individual boulder comparison — matched pairs side by side

In [ ]:
# Find matched pairs: YOLO detection ↔ SAM2 detection with IoU ≥ 0.25
# Use spatial join to get candidate overlapping pairs, then compute IoU

gdf_yolo_tmp = gdf_yolo[["geometry", "poly_area"]].copy()
gdf_sam2_tmp = gdf_sam2[["geometry", "poly_area"]].copy()
gdf_sam2_tmp["sam2_idx"] = gdf_sam2_tmp.index

joined = gpd.sjoin(gdf_yolo_tmp, gdf_sam2_tmp, how="inner", predicate="intersects")
joined = joined.rename(columns={"poly_area_left": "yolo_area",
                                  "poly_area_right": "sam2_area"})
joined["yolo_idx"] = joined.index

def iou(g1, g2):
    g1 = g1.buffer(0)
    g2 = g2.buffer(0)
    inter = g1.intersection(g2).area
    uni   = g1.union(g2).area
    return inter / uni if uni > 0 else 0.0

# Compute IoU for each candidate pair (sample first to keep it fast)
joined_sample = joined.sample(min(5000, len(joined)), random_state=42)
joined_sample["iou"] = [
    iou(gdf_yolo.loc[r.yolo_idx, "geometry"],
        gdf_sam2.loc[r.sam2_idx, "geometry"])
    for _, r in joined_sample.iterrows()
]

matched = joined_sample[joined_sample["iou"] >= 0.25].drop_duplicates(subset="yolo_idx")
print(f"Matched pairs with IoU ≥ 0.25: {len(matched)}")

In [ ]:
# Plot 16 matched pairs: YOLO (left) vs SAM2 (right) for each boulder
n_show = 16
sample = matched.sample(min(n_show, len(matched)), random_state=0)
pad_m  = 4  # metres of padding around each boulder

fig, axes = plt.subplots(n_show, 2, figsize=(6, n_show * 2))
if n_show == 1:
    axes = axes[np.newaxis, :]

with rasterio.open(in_raster) as src:
    for plot_i, (_, pair) in enumerate(sample.iterrows()):
        yolo_geom = gdf_yolo.loc[pair["yolo_idx"], "geometry"]
        sam2_geom = gdf_sam2.loc[pair["sam2_idx"], "geometry"]

        minx = min(yolo_geom.bounds[0], sam2_geom.bounds[0]) - pad_m
        miny = min(yolo_geom.bounds[1], sam2_geom.bounds[1]) - pad_m
        maxx = max(yolo_geom.bounds[2], sam2_geom.bounds[2]) + pad_m
        maxy = max(yolo_geom.bounds[3], sam2_geom.bounds[3]) + pad_m

        try:
            window = from_bounds(minx, miny, maxx, maxy, src.transform)
            img = src.read(1, window=window).astype(float)
            p2, p98 = np.percentile(img, [2, 98])
            img_norm = np.clip((img - p2) / (p98 - p2 + 1e-6), 0, 1)
            wt = src.window_transform(window)

            for col_i, (geom, color, title) in enumerate([
                (yolo_geom, "steelblue", f"YOLOv8  {pair['yolo_area']:.1f} m²"),
                (sam2_geom, "tomato",    f"SAM2    {pair['sam2_area']:.1f} m²"),
            ]):
                ax = axes[plot_i, col_i]
                ax.imshow(img_norm, cmap="gray", origin="upper")
                coords = np.array(geom.exterior.coords)
                px = np.array([~wt * (x, y) for x, y in coords])
                ax.plot(px[:, 0], px[:, 1], color=color, linewidth=1.5)
                ax.set_title(title, fontsize=7)
                ax.axis("off")
        except Exception:
            axes[plot_i, 0].axis("off")
            axes[plot_i, 1].axis("off")

plt.suptitle("Individual boulder comparison\nYOLOv8 (blue) vs YOLOv8+SAM2 (red)",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig("qual_individual_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 2: Mask Regularity Metrics

- **Compactness** = 4π·area / perimeter² — 1 for a perfect circle, lower for jagged or elongated shapes
- **Convexity** = area / convex_hull_area — 1 for fully convex shapes, lower for concave/irregular ones

YOLOv8 staircase masks should have lower compactness and convexity than SAM2 smooth masks.

In [ ]:
def compute_regularity(gdf):
    records = []
    for geom in tqdm(gdf.geometry):
        area      = geom.area
        perim     = geom.length
        compact   = (4 * np.pi * area) / (perim ** 2) if perim > 0 else np.nan
        ch_area   = geom.convex_hull.area
        convexity = area / ch_area if ch_area > 0 else np.nan
        records.append({"area": area, "compactness": compact, "convexity": convexity})
    return pd.DataFrame(records)

print("YOLO regularity...")
df_yolo_reg = compute_regularity(gdf_yolo)
print("SAM2 regularity...")
df_sam2_reg = compute_regularity(gdf_sam2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, xlabel in zip(axes,
    ["compactness", "convexity"],
    ["Compactness  (4π·area / perimeter²)",
     "Convexity  (area / convex hull area)"]):

    y_med = df_yolo_reg[metric].median()
    s_med = df_sam2_reg[metric].median()

    ax.hist(df_yolo_reg[metric].dropna(), bins=60, alpha=0.6,
            color="steelblue", label="YOLOv8",      density=True)
    ax.hist(df_sam2_reg[metric].dropna(), bins=60, alpha=0.6,
            color="tomato",    label="YOLOv8+SAM2", density=True)
    ax.axvline(y_med, color="steelblue", linestyle="--",
               label=f"YOLO median = {y_med:.3f}")
    ax.axvline(s_med, color="tomato",    linestyle="--",
               label=f"SAM2 median = {s_med:.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

axes[0].set_title("Mask compactness\n(1 = perfect circle)")
axes[1].set_title("Mask convexity\n(1 = fully convex)")

plt.suptitle("Mask regularity metrics — YOLOv8 vs YOLOv8+SAM2", fontsize=13)
plt.tight_layout()
plt.savefig("exp_mask_regularity.png", dpi=150)
plt.show()

print(f"YOLO  compactness: {df_yolo_reg['compactness'].median():.4f}  "
      f"convexity: {df_yolo_reg['convexity'].median():.4f}")
print(f"SAM2  compactness: {df_sam2_reg['compactness'].median():.4f}  "
      f"convexity: {df_sam2_reg['convexity'].median():.4f}")